# WebSocket streaming — real-time run events

Almost every SDK call is a plain **request/response over HTTPS**: you send one request, the
server sends one response, the connection is done. One family of calls is different.
**`client.runs.stream()`** opens a **persistent, bidirectional WebSocket** to the workflow
server and keeps it open for the whole run:

```
wss://<host>/workflows/v1/workflow-runs/ws
```

Over that single socket:

- **client → server** — the SDK sends one opening frame: a JSON `WorkflowRunInput`
  (`workflow_id`, `command`, optional variables / per-turn inputs) that kicks the run off.
- **server → client** — the server then *pushes* a live stream of `RunEvent` frames as the
  run executes (node starts, assistant responses, edge traversals, …) until it finishes and
  closes the socket.

That two-way traffic on one long-lived connection is what makes it a bidirectional WebSocket
rather than a series of independent HTTP calls — you see events *as they happen* instead of
polling for a final result.

This notebook:
1. Shows the WebSocket endpoint and the opening frame the SDK sends
2. Streams a run's events live with `client.runs.stream()` (async context manager)
3. Inspects the `RunEvent` structure and the event types seen on the wire
4. Contrasts it with the ordinary HTTP request/response path (`client.runs.execute()`)

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import nest_asyncio

from interactly import AsyncWorkflowClient, WorkflowCommand
from interactly.configs import (
    DirectEdgeConfig,
    SayStaticMessageNodeConfig,
    StaticMessagesConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
)
from interactly.types.workflows.workflow import Workflow

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()

# A tiny two-node workflow we can stream events from (greeting -> goodbye).
_greeting = SayStaticMessageNodeConfig(
    name="Greeting",
    is_start=True,
    static_messages_config=StaticMessagesConfig(static_messages=["Hello! Streaming events now."]),
)
_goodbye = SayStaticMessageNodeConfig(
    name="Goodbye",
    static_messages_config=StaticMessagesConfig(static_messages=["All done — goodbye!"]),
)
_config = WorkflowConfigFullyHydrated(
    workflow_config=WorkflowConfig(name="WebSocket Streaming Notebook", category="System Examples"),
    node_configs=[_greeting, _goodbye],
    edge_configs=[
        DirectEdgeConfig(
            source_node_logical_id=_greeting.logical_id,
            destination_node_logical_id=_goodbye.logical_id,
            name="greeting->goodbye",
        )
    ],
)
workflow: Workflow = await client.workflows.create_from_config(_config)
WORKFLOW_ID = workflow.id

print("Connected to", client._base_url)
print("Created workflow", WORKFLOW_ID)

## The WebSocket endpoint and the opening frame

`client.runs.stream()` derives a `wss://` URL from your base URL and connects to the
`/v1/workflow-runs/ws` endpoint. Before any events arrive, the SDK sends exactly one frame —
the run command, shaped as a `WorkflowRunInput`. Everything after that is server → client.

The two cells below just *show* the URL and a representative opening frame so the two
directions are concrete; the next section actually opens the socket.

In [ ]:
import json

# The SDK flips the http(s) base URL to ws(s) and appends the run-stream path.
ws_url = client._ws_base_url() + "/v1/workflow-runs/ws"
print("WebSocket URL :", ws_url)

# The single frame the client sends first (client -> server) to start the run.
opening_frame = {
    "workflow_id": WORKFLOW_ID,
    "command": WorkflowCommand.START.value,
    "run_by": "api",
    # A default thread is seeded so a bare START is a valid run; a full turn would
    # carry its user message here under thread_to_node_inputs.
    "thread_to_node_inputs": {"0": {"node_run_inputs": []}},
}
print("Opening frame :", json.dumps(opening_frame, indent=2))

## 1. Stream a run's events live

`client.runs.stream(...)` returns an **async context manager**. Entering it (`async with`)
opens the WebSocket and sends the opening frame; iterating it (`async for`) yields each
`RunEvent` the server pushes over that *same* connection. The socket stays open for the whole
loop and closes automatically when the `async with` block exits.

`stream()` also accepts `dynamic_variables` / `runtime_variables`, or a full typed `run_input`
(a per-turn `WorkflowRunInput`) — all sent in that opening frame. To drive a multi-turn chat,
see [`12_runtime_handles.ipynb`](12_runtime_handles.ipynb).

In [ ]:
events_received = []

async with client.runs.stream(
    workflow_id=WORKFLOW_ID,
    command=WorkflowCommand.START,
) as stream:
    async for event in stream:
        events_received.append(event)
        text = event.output if event.output else ""
        print(f"  [{event.type:24s}] node={event.node_id or '—'}  {text}")

        if event.is_terminal():
            print(f"\n→ Terminal event ({event.type}); the server now closes the socket.")
            break

print(f"\nTotal events received over the one connection: {len(events_received)}")

## 2. The `RunEvent` structure

Each frame is parsed into a `RunEvent` with a `type`, an optional `node_id`, an `output`
payload (e.g. assistant text), a run-level `status`, and (on failure) an `error`. Unknown
event types are accepted as-is, so new server event types never break the SDK. Use
`is_terminal()` to detect the end of the run (`end_workflow` / `workflow_error`).

In [ ]:
# Show a couple of representative frames: the first, and the terminal one.
first = events_received[0]
print("first event")
print(f"  type       : {first.type}")
print(f"  node_id    : {first.node_id}")
print(f"  output     : {first.output}")
print(f"  status     : {first.status}")
print(f"  is_terminal: {first.is_terminal()}")

last = events_received[-1]
print("\nlast event")
print(f"  type       : {last.type}")
print(f"  is_terminal: {last.is_terminal()}")

## 3. Event types seen on the stream

The exact set depends on the workflow, but a simple say-node run emits frames like these
(the list is **not** exhaustive — the server may emit others, which pass through untouched):

| Type | Meaning |
|------|---------|
| `run_ack` | handshake acknowledgement (synthetic first frame) |
| `start_node_run` / `end_node_run` | a node began / finished executing |
| `say_static` / `assistant_response` | a message the node produced (`output` holds the text) |
| `direct_edge` / `conditional_edge` | an edge transition was taken |
| `workflow_show_state` | a snapshot of run state |
| `end_thread` / `end_workflow_iteration` | a thread / iteration finished |
| `end_workflow` | the run finished (**terminal**) |
| `workflow_error` | the run failed (**terminal**) |

In [ ]:
from collections import Counter

counts = Counter(e.type for e in events_received)
for event_type, count in counts.most_common():
    print(f"  {event_type:30s}  {count}")

## 4. Contrast: the ordinary HTTP path (`execute()`)

When you only need the *final* result — not a live feed — use `client.runs.execute()`. That
is a normal **HTTP POST** (request in, one response out); no persistent socket. It returns an
`InteractiveRunResponse`; echo its `run_id` on each subsequent turn until `is_completed`.

Rule of thumb: **`stream()` (WebSocket)** when you want events as they happen; **`execute()`
(HTTP)** when you just want the outcome of a turn.

In [ ]:
from interactly.types.runs.interactive_run import InteractiveRunResponse

result: InteractiveRunResponse = await client.runs.execute(WORKFLOW_ID)

print(f"Run id            : {result.run_id}")
print(f"Status            : {result.status}")
print(f"Turn number       : {result.turn_number}")
print(f"Waiting for input : {result.is_waiting_for_input}")
print(f"Completed         : {result.is_completed}")

## Cleanup

Delete the demo workflow and close the client.

In [ ]:
await client.workflows.delete(WORKFLOW_ID)
await client.close()
print("Cleaned up workflow", WORKFLOW_ID)

## See also

- [`12_runtime_handles.ipynb`](12_runtime_handles.ipynb) — multi-turn chat via a runtime handle
- [`14_pagination_and_filtering.ipynb`](14_pagination_and_filtering.ipynb) — listing past runs and their events